In [ ]:
%pip install /Workspace/Users/neil.braun@mirakl.com/.bundle/fast-gnn-benchmark/dev/files
dbutils.library.restartPython()

In [ ]:
from fast_gnn_benchmark.data.dataset.coview_mdm import CoViewMDMDataset

import os
import math
from collections import defaultdict

import faiss
import numpy as np
import torch

import boto3, json
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from IPython.display import display

In [ ]:
from datetime import datetime

# basculer sur coview-mdm-cosine pour evaluer la tete cosine nue (gae_gcn_coview_mdm_cosine.yml)
ckpt_dir = "/dbfs/tmp/nbraun/checkpoints/coview-mdm-mlp-cosine"

def list_checkpoints(ckpt_dir: str) -> list[tuple[float, str]]:

    files = [
        (os.path.getmtime(os.path.join(ckpt_dir, f)),f)
        for f in os.listdir(ckpt_dir)
        if f.endswith(".ckpt")
    ]

    return sorted(files, reverse=True)

In [ ]:
for mtime, fname in list_checkpoints(ckpt_dir):
    print(f"{datetime.fromtimestamp(mtime):%Y-%m-%d %H:%M:%S} {fname}")

In [ ]:
from fast_gnn_benchmark.models.link_prediction import LinkPredictionModel


def load_best_checkpoint(
    ckpt_dir: str, checkpoint: str, device: str
) -> tuple[LinkPredictionModel, str, object]:
    """Charge `checkpoint` si un chemin est fourni, sinon le plus recent de `ckpt_dir`.

    contrat des tetes indexables: project() renvoie un vecteur unitaire par noeud dont le produit
    scalaire est le score. hadamard_mlp ne l'a pas, son score n'etant pas decomposable.
    """
    if checkpoint == "latest":
        checkpoints = list_checkpoints(ckpt_dir)
        assert checkpoints, f"aucun .ckpt dans {ckpt_dir}"
        checkpoint_path = os.path.join(ckpt_dir, checkpoints[0][1])
    else:
        checkpoint_path = checkpoint

    print(f"checkpoint: {checkpoint_path}")

    raw_ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

    model = LinkPredictionModel.load_from_checkpoint(checkpoint_path, map_location=device, weights_only=False)
    model.eval()
    model.to(device)

    print("epoch:", raw_ckpt["epoch"])
    print("global_step:", raw_ckpt["global_step"])
    print("nb params:", sum(p.numel() for p in model.parameters()))

    head = model.model.classifier
    assert hasattr(head, "project"), (
        f"head {type(head).__name__} non indexable -- ce notebook exige une tete decomposable "
        f"(cosine_similarity ou mlp_cosine) (checkpoint: {checkpoint_path})"
    )
    print("head:", type(head).__name__)

    return model, checkpoint_path, head


device = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT = "latest"

model, checkpoint_path, head = load_best_checkpoint(ckpt_dir, CHECKPOINT, device)

## Dataset Loading

In [ ]:
def load_prototype_artifacts(bucket: str, prefix: str) -> tuple[CoViewMDMDataset, dict, dict, dict]:
    """Charge le dataset et les mappings prototype depuis S3.

    Artefacts prototype : exec2code y couvre toutes les executions du split, donc les exec_code
    diffèrent de ceux du pipeline principal. Ne jamais melanger avec data.pt / exec_mappings.json.
    """
    dataset = CoViewMDMDataset(bucket=bucket, s3_key=f"{prefix}/data_prototype.pt")

    s3 = boto3.client("s3")

    def load_json_from_s3(key: str) -> dict:
        return json.loads(s3.get_object(Bucket=bucket, Key=key)["Body"].read())

    node2idx_raw = load_json_from_s3(f"{prefix}/node2idx_prototype.json")
    node2idx = {int(k): v for k, v in node2idx_raw.items()}
    idx2node = {v: k for k, v in node2idx.items()}

    exec_mappings = load_json_from_s3(f"{prefix}/exec_mappings_prototype.json")

    return dataset, node2idx, idx2node, exec_mappings


BUCKET = "mirakl-data-science-tmp2"
PREFIX = "nbraun/datasets/coview-mdm"

dataset, node2idx, idx2node, exec_mappings = load_prototype_artifacts(BUCKET, PREFIX)

print(f"num_nodes: {dataset.num_nodes}")
print(f"node2idx entries: {len(node2idx)}")
for split in ["train", "val", "test"]:
    print(f"{split}: {len(exec_mappings[split]['code2exec'])} executions")

## Load prototype tables

In [ ]:
prod_results_val_prototype = spark.read.parquet(
    f"s3://{BUCKET}/{PREFIX}/prod_results_val_prototype.parquet"
)
sessions_raw_val_prototype = spark.read.parquet(
    f"s3://{BUCKET}/{PREFIX}/sessions_raw_val_prototype.parquet"
)

print(f"prod_results_val_prototype: {prod_results_val_prototype.count()} triggers")
prod_results_val_prototype.printSchema()
display(prod_results_val_prototype.limit(1))

print(f"sessions_raw_val_prototype: {sessions_raw_val_prototype.count()} triggers")
display(sessions_raw_val_prototype.limit(1))

## We calculate the embeddings for every node

In [ ]:
import torch
from torch_geometric.data import Data
from torch_geometric.transforms import ToSparseTensor


def compute_node_embeddings(
    dataset: CoViewMDMDataset, model: LinkPredictionModel, head, device: str
) -> tuple[torch.Tensor, np.ndarray]:
    """Forward pass GNN complet (embedder + backbone), puis projection dans l'espace indexable par
    faiss : head.project() renvoie un vecteur unitaire par noeud dont le produit scalaire est le
    score (normalisation seule pour la tete cosine nue, tour MLP puis normalisation pour mlp_cosine).
    """
    N = dataset.num_nodes
    adj_t = ToSparseTensor()(Data(edge_index=dataset.data.edge_index, num_nodes=N)).adj_t.to(device)

    for m in model.model.backbone.modules():
        if hasattr(m, "_cached_edge_index"):
            m._cached_edge_index = None
        if hasattr(m, "_cached_adj_t"):
            m._cached_adj_t = None

    torch.cuda.empty_cache()

    model.eval()

    with torch.no_grad():
        x = model.model.embedder(dataset.data.x.to(device))
        x = model.model.backbone(x, adj_t)
        x_normalized = head.project(x)
        xn_np = np.ascontiguousarray(x_normalized.detach().cpu().numpy(), dtype="float32")

    return x_normalized, xn_np


x_normalized, xn_np = compute_node_embeddings(dataset, model, head, device)

print(f"x shape: {tuple(x_normalized.shape)}")
print(f"norm after projection: {x_normalized.norm(dim=-1)[:5]}")

In [ ]:
def extract_all_triggers(prod_results: DataFrame) -> list[dict]:
    """Tous les triggers de la table prod prototype.

    Remplace extract_triggers (qui partait de val_split["edge"], donc des seules executions ayant
    au moins un positif dans le top-12 prod) et extract_negative_only_triggers. Le tenseur d'edges
    n'est plus utilise ici: la population d'analyse est definie independamment de l'etiquetage.
    """
    rows = (
        prod_results
        .select("exec_code", "trigger_internal_id")
        .dropDuplicates(["exec_code"])
        .collect()
    )

    triggers = []
    skipped_no_node = 0

    for row in rows:
        trigger_internal_id = int(row["trigger_internal_id"])

        if trigger_internal_id not in node2idx:
            skipped_no_node += 1
            continue

        triggers.append({
            "exec_code": row["exec_code"],
            "trigger_internal_id": trigger_internal_id,
            "trigger_node_id": node2idx[trigger_internal_id],
        })

    print(f"triggers hors graphe (produit sans node_id): {skipped_no_node}/{len(rows)}")

    return triggers


triggers = extract_all_triggers(prod_results_val_prototype)

print(f"{len(triggers)} triggers a scorer")

We consider that the triggers must be in the graph (so in the active catalog) -> we count the triggers that do not correspond to a node in the graph

We extract the t2s_best_fitting_category for the triggers found before

In [ ]:
def extract_categories(triggers: list[dict]) -> list[dict]:

    unique_internal_ids = {t["trigger_internal_id"] for t in triggers}

    df_trigger_ids = spark.createDataFrame(
        [(str(i),) for i in unique_internal_ids], schema="internalId string"
    )

    df_categories = (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(F.col("customer_short_name") == "maisons-du-monde")
        .join(F.broadcast(df_trigger_ids), on="internalId", how="left_semi")
        .select("internalId", F.col("t2s_best_fitting_category")[0].alias("category"))
        .dropDuplicates(["internalId"])
        .collect()
    )

    category_by_internal_id = {
        int(row["internalId"]): row["category"]
        for row in df_categories
        if row["category"] is not None
    }

    for trigger in triggers:
        trigger["category"] = category_by_internal_id.get(trigger["trigger_internal_id"])

    return triggers

triggers = extract_categories(triggers)

missing = sum(1 for t in triggers if t["category"] is None)
print(f"triggers sans categorie: {missing}/{len(triggers)}")

In [ ]:
def build_candidates_by_category(triggers: list[dict]) -> dict:

    unique_categories = {t["category"] for t in triggers if t["category"] is not None}

    df_category_products = (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(
            F.col("t2s_best_fitting_category")[0].isin(list(unique_categories))
            & (F.col("customer_short_name") == "maisons-du-monde")
        )
        .select("internalId", F.col("t2s_best_fitting_category")[0].alias("category"))
        .dropDuplicates(["internalId"])
        .collect()
    )

    internal_ids_by_category = {c: [] for c in unique_categories}
    for row in df_category_products:
        internal_ids_by_category[row["category"]].append(int(row["internalId"]))

    candidate_node_ids_by_category = {}
    for category, internal_ids in internal_ids_by_category.items():
        node_ids = torch.tensor([node2idx[i] for i in internal_ids if i in node2idx], dtype=torch.long)
        candidate_node_ids_by_category[category] = torch.unique(node_ids)  # plusieurs internalId peuvent partager le même node_id

    return candidate_node_ids_by_category

candidates_by_category = build_candidates_by_category(triggers)

pools_np = {
    cat: pool.numpy().astype(np.int64)
    for cat, pool in candidates_by_category.items()
}

pool_sizes = [c.numel() for c in candidates_by_category.values()]
print(f"nombre de catégories: {len(candidates_by_category)}")
print(f"taille moyenne du pool de candidats par catégorie: {sum(pool_sizes) / len(pool_sizes):.0f}")

In [ ]:
def build_flat_index(vectors: np.ndarray) -> faiss.Index:
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vectors)
    return index

def build_ivf_index(vectors: np.ndarray, nlist: int, nprobe: int) -> faiss.Index:
    dim = vectors.shape[1]
    quantizer = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
    index.cp.seed = 42

    index.train(vectors)
    index.add(vectors)

    index.nprobe = nprobe
    return index


indexes_flat_cat = {
    cat: build_flat_index(np.ascontiguousarray(xn_np[pool]))
    for cat, pool in pools_np.items()
}

print(f"index construits: {len(indexes_flat_cat)}")
print(f"exemple ({next(iter(indexes_flat_cat))}): {next(iter(indexes_flat_cat.values())).ntotal} vecteurs")

by_cat = defaultdict(list)
for i, t in enumerate(triggers):
    if t["category"] in pools_np:
        by_cat[t["category"]].append(i)

In [ ]:
def run_retrieval(index: faiss.Index, pool: np.ndarray, query_node_ids: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:

    Q = np.ascontiguousarray(xn_np[query_node_ids])

    fetch = k+1
    D, I = index.search(Q, min(fetch, index.ntotal))

    ids = np.where(I >= 0, pool[np.clip(I, 0, None)], -1)
    keep = (ids >= 0) & (ids != query_node_ids[:, None])

    order = np.argsort(~keep, axis=1, kind="stable")
    ids_sorted = np.take_along_axis(ids, order, axis=1)
    scores_sorted = np.take_along_axis(D, order, axis=1).astype(np.float32)

    n_keep = keep.sum(axis=1)
    col_idx = np.arange(ids_sorted.shape[1])
    invalid = col_idx[None, :] >= n_keep[:, None]
    ids_sorted = np.where(invalid, -1, ids_sorted)
    scores_sorted = np.where(invalid, np.nan, scores_sorted)

    top_ids = ids_sorted[:, :k]
    top_scores = scores_sorted[:, :k]

    if top_ids.shape[1] < k:
        pad = k - top_ids.shape[1]
        top_ids = np.concatenate([top_ids, np.full((len(top_ids), pad), -1, dtype=np.int64)], axis=1)
        top_scores = np.concatenate([top_scores, np.full((len(top_scores), pad), np.nan, dtype=np.float32)], axis=1)

    return top_ids, top_scores

In [ ]:
def run_inference(triggers, indexes_flat_cat, pools_np, by_cat, k):
    n_triggers = len(triggers)
    top_ids = np.full((n_triggers, k), -1, dtype=np.int64)
    top_scores = np.full((n_triggers, k), np.nan, dtype=np.float32)

    for cat, positions in by_cat.items():
        query_node_ids = np.array([triggers[i]["trigger_node_id"] for i in positions], dtype=np.int64)
        ids, scores = run_retrieval(indexes_flat_cat[cat], pools_np[cat], query_node_ids, k)
        top_ids[positions] = ids
        top_scores[positions] = scores

    return top_ids, top_scores


top_ids, top_scores = run_inference(triggers, indexes_flat_cat, pools_np, by_cat, k=12)

print(top_ids.shape, top_scores.shape)
print(f"triggers sans reponse: {(top_ids[:, 0] == -1).sum()}/{len(triggers)}")

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, DoubleType

BATCH_TRIGGERS = 20_000
MODEL_ROWS_PATH_ANN = f"s3://{BUCKET}/{PREFIX}/_staging_model_rows_ann_val_prototype.parquet"
OUTPUT_ANN = f"s3://{BUCKET}/{PREFIX}/model_results_ann_val_prototype.parquet"

model_rows_schema = StructType([
    StructField("exec_code", LongType(), True),
    StructField("internal_id", LongType(), True),
    StructField("score", DoubleType(), True),
    StructField("rank", IntegerType(), True),
])

keys_schema = StructType([
    StructField("exec_code", LongType(), True),
    StructField("trigger_internal_id", LongType(), True),
])

EMPTY_PRODUCTS = F.array().cast("array<struct<internal_id:bigint,score:double,rank:int>>")


def stage_model_rows_ann(triggers, top_ids, top_scores, path, batch_triggers=BATCH_TRIGGERS):
    mode = "overwrite"
    n = len(triggers)

    for start in range(0, n, batch_triggers):
        end = min(start + batch_triggers, n)
        rows = [
            (triggers[i]["exec_code"], int(idx2node[int(nid)]), float(sc), rank)
            for i in range(start, end)
            for rank, (nid, sc) in enumerate(zip(top_ids[i], top_scores[i]), start=1)
            if nid >= 0
        ]
        if not rows:
            continue

        (
            spark.createDataFrame(rows, schema=model_rows_schema, verifySchema=False)
            .write.mode(mode).parquet(path)
        )
        print(f"  lot {start}-{end}: {len(rows)} lignes")
        mode = "append"

    if mode == "overwrite":
        spark.createDataFrame([], schema=model_rows_schema).write.mode("overwrite").parquet(path)
        print("  aucun trigger score, parquet vide ecrit")


def build_model_results_ann(triggers, top_ids, top_scores, sessions_raw, rows_path):

    stage_model_rows_ann(triggers, top_ids, top_scores, rows_path)

    df_keys = spark.createDataFrame(
        [(t["exec_code"], t["trigger_internal_id"]) for t in triggers],
        schema=keys_schema,
        verifySchema=False,
    )

    df_products = (
        spark.read.parquet(rows_path)
        .groupBy("exec_code")
        .agg(F.sort_array(F.collect_list(F.struct("rank", "internal_id", "score"))).alias("ranked"))
        .withColumn(
            "products_returned",
            F.transform(
                "ranked",
                lambda r: F.struct(
                    r["internal_id"].alias("internal_id"),
                    r["score"].alias("score"),
                    r["rank"].alias("rank"),
                ),
            ),
        )
        .select("exec_code", "products_returned")
    )

    session_ids_by_exec_code = (
        sessions_raw
        .select("exec_code", F.col("session_products.internal_id").alias("session_ids"))
        .dropDuplicates(["exec_code"])
    )

    return (
        df_keys
        .join(df_products, on="exec_code", how="left")
        .join(session_ids_by_exec_code, on="exec_code", how="left")
        .withColumn("products_returned", F.coalesce(F.col("products_returned"), EMPTY_PRODUCTS))
        .withColumn("session_ids", F.coalesce(F.col("session_ids"), F.array().cast("array<bigint>")))
        .withColumn(
            "positives",
            F.filter("products_returned", lambda p: F.array_contains(F.col("session_ids"), p["internal_id"])),
        )
        .withColumn(
            "negatives",
            F.filter("products_returned", lambda p: ~F.array_contains(F.col("session_ids"), p["internal_id"])),
        )
        .select("exec_code", "trigger_internal_id", "positives", "negatives", "products_returned")
    )


model_results_ann_val_prototype = build_model_results_ann(
    triggers, top_ids, top_scores, sessions_raw_val_prototype, MODEL_ROWS_PATH_ANN
).cache()

print(f"model_results_ann_val_prototype: {model_results_ann_val_prototype.count()} triggers")

model_results_ann_val_prototype.printSchema()
display(model_results_ann_val_prototype.limit(1))

In [ ]:
model_results_ann_val_prototype.write.mode("overwrite").parquet(OUTPUT_ANN)

print("model_results_ann_val_prototype.parquet uploaded")

In [ ]:
SAMPLE_SIZE = 500
TOP_K = 12
sample_positions = [i for i in range(len(triggers)) if triggers[i]["category"] is not None][:SAMPLE_SIZE]

def run_inference_torch_reference(positions, top_k=TOP_K):
    results = {}
    with torch.no_grad():
        for i in positions:
            t = triggers[i]
            candidates = candidates_by_category[t["category"]]
            candidates = candidates[candidates != t["trigger_node_id"]]
            if candidates.numel() == 0:
                continue
            target_edges = torch.stack([
                torch.full_like(candidates, t["trigger_node_id"]),
                candidates,
            ]).to(device)
            # produit scalaire explicite sur les vecteurs deja projetes: c'est exactement ce que
            # calcule faiss. Passer x_normalized a classifier() reappliquerait project() par-dessus,
            # invisible pour la cosine nue (renormaliser est idempotent) mais faux pour mlp_cosine.
            logits = (x_normalized[target_edges[0]] * x_normalized[target_edges[1]]).sum(dim=-1)
            k = min(top_k, logits.numel())
            top_logits, top_pos = torch.topk(logits, k)
            top_node_ids = candidates[top_pos.cpu()]
            results[i] = (top_node_ids.cpu().numpy(), top_logits.cpu().numpy())
    return results

ref = run_inference_torch_reference(sample_positions)

n_exact, n_diff = 0, 0
for i, (ref_ids, ref_scores) in ref.items():
    faiss_valid = top_ids[i][top_ids[i] >= 0]
    faiss_valid_scores = top_scores[i][top_ids[i] >= 0]

    if set(ref_ids.tolist()) == set(faiss_valid.tolist()):
        assert np.allclose(np.sort(ref_scores)[::-1], np.sort(faiss_valid_scores)[::-1], atol=1e-5)
        n_exact += 1
    else:
        n_diff += 1
        print(f"trigger {i}: ref={sorted(ref_ids.tolist())} faiss={sorted(faiss_valid.tolist())}")

print(f"{n_exact}/{len(ref)} identiques, {n_diff} a inspecter")

In [ ]:
MIN_POOL_FOR_ANN = 2048
NPROBE = 8

indexes_ivf_cat = {}
for cat, pool in pools_np.items():
    n = len(pool)
    if n < MIN_POOL_FOR_ANN:
        indexes_ivf_cat[cat] = indexes_flat_cat[cat]
        continue
    nlist = max(1, min(n // 39, int(4 * math.sqrt(n))))
    indexes_ivf_cat[cat] = build_ivf_index(np.ascontiguousarray(xn_np[pool]), nlist=nlist, nprobe=NPROBE)

n_ivf = sum(1 for cat in pools_np if len(pools_np[cat]) >= MIN_POOL_FOR_ANN)
print(f"index ivf construits: {n_ivf}/{len(pools_np)} (reste en flat sous {MIN_POOL_FOR_ANN})")

In [ ]:
top_ids_ann, top_scores_ann = run_inference(triggers, indexes_ivf_cat, pools_np, by_cat, k=12)

print(top_ids_ann.shape, top_scores_ann.shape)
print(f"triggers sans reponse: {(top_ids_ann[:, 0] == -1).sum()}/{len(triggers)}")

In [ ]:
valid = top_ids >= 0
inter = (top_ids_ann[:, :, None] == top_ids[:, None, :]) & valid[:, None, :]
recall_per_trigger = inter.any(1).sum(1) / np.maximum(valid.sum(1), 1)

print(f"recall@12 moyen (ivf vs exact): {recall_per_trigger.mean():.4f}")
print(f"triggers a recall < 1.0: {(recall_per_trigger < 1.0).sum()}/{len(recall_per_trigger)}")

In [ ]:
model_results_ann_val_prototype = build_model_results_ann(
    triggers, top_ids_ann, top_scores_ann, sessions_raw_val_prototype, MODEL_ROWS_PATH_ANN
).cache()

print(f"model_results_ann_val_prototype: {model_results_ann_val_prototype.count()} triggers")

model_results_ann_val_prototype.printSchema()
display(model_results_ann_val_prototype.limit(1))

In [ ]:
model_results_ann_val_prototype.write.mode("overwrite").parquet(OUTPUT_ANN)

print("model_results_ann_val_prototype.parquet uploaded (ivf)")